### Standard imports

In [4]:
import pandas as pd

### The 6M+ games dataset can't fit into RAM, so we're loading it in chunks.

The original dataset was too large to upload to GitHub, even with Git LFS. If you need the raw dataset, you can download it from the original Kaggle source:

> **Source:** [Kaggle - Chess Games Dataset](https://www.kaggle.com/datasets/arevel/chess-games/data)

In [6]:
raw_df = pd.read_csv("../../data/raw/chess_games.csv", chunksize=10_000)

chunk = next(raw_df)
print(chunk.columns)

Index(['Event', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo',
       'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening',
       'TimeControl', 'Termination', 'AN'],
      dtype='str')


### Dataset Columns

- **Event:** Game type.
- **White:** White's ID.
- **Black:** Black's ID.
- **Result:** Game result (`1-0` White wins, `0-1` Black wins).
- **UTCDate:** UTC date.
- **UTCTime:** UTC time.
- **WhiteElo:** White's ELO.
- **BlackElo:** Black's ELO.
- **WhiteRatingDiff:** White's rating points difference after the game.
- **BlackRatingDiff:** Black's rating points difference after the game.
- **ECO:** Opening in [ECO encoding](https://www.365chess.com/eco.php).
- **Opening:** Opening name.
- **TimeControl:** Time of the game for each player in seconds. The number after the increment is the number of seconds before the player's clock starts ticking on each turn.
- **Termination:** Reason for the game's end.
- **AN:** Moves in Movetext format.

### Data filtering

### Filtering Games by Average ELO and Time Forfeit

Filtering out games with an average ELO below 2650 and games that ended by time forfeit to reduce the dataset size and memory usage.

In [ ]:
input_path = "../../data/raw/chess_games.csv"
output_path = "../../data/processed/chess_games_filtered.csv"

first_chunk = True

for chunk in pd.read_csv(input_path, chunksize=10_000):
    chunk["avg_elo"] = (chunk["WhiteElo"] + chunk["BlackElo"]) / 2

    filtered = chunk[
        (chunk["avg_elo"] >= 2650) &
        (chunk["Termination"] != "Time forfeit")
    ]

    filtered.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print("Done!")

Done!


### Dataset Inspection

Inspecting the filtered CSV in chunks to estimate the total games, average moves per game, and total positions without loading the entire dataset into RAM.

In [10]:
path = "../../data/processed/chess_games_filtered.csv"

total_games = 0
sample_games = 0
sample_moves = 0

for i, chunk in enumerate(pd.read_csv(path, chunksize=10_000)):
    total_games += len(chunk)

    # Estimate average moves from a small sample
    if i == 0:
        sample = chunk["AN"].dropna().head(1000)
        sample_moves = sample.str.count(r"\d+\.").mean()

print("Total games:", total_games)
print("Estimated average moves/game:", round(sample_moves, 1))
print("Estimated positions:", round(total_games * sample_moves))

Total games: 6860
Estimated average moves/game: 64.8
Estimated positions: 444679


### Using the Filtered Dataset as the Main Source

With the dataset now smaller and manageable in RAM, `chess_games_filtered.csv` will be used as the main source file for further processing and analysis.